DSCI 552 Homework 4

Name: Brynn Dafoe GitHub Username: brynndafoe02 USD ID: 3109-6692-10

HOMEWORK 3 PORTION NEEDED FOR HOMEWORK 4:

In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, mean_squared_error, r2_score, mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler

from itertools import combinations
import seaborn as sns

import statsmodels.api as sm
import statsmodels.formula.api as smf

from pathlib import Path
import re

In [16]:
path_to_AReM = Path("../data/AReM")
column_names = ["time", "avg_rss12", "var_rss12", "avg_rss13", "var_rss13", "avg_rss23", "var_rss23"]

training_dfs = {}
testing_dfs = {}

for data_dir in path_to_AReM.iterdir():
    if data_dir.is_dir():
        dir_name = data_dir.name
            
        for file in data_dir.iterdir():
            file_name = Path(file).stem
            file_num = int("".join(re.findall(r"\d", file_name)))

            key = f"{dir_name}_{file_name}"
            
            if (dir_name == "bending1") or (dir_name == "bending2"):
                if dir_name == "bending1":
                    df = pd.read_csv(file, skiprows=5, names=column_names)
                else:
                    df = pd.read_csv(file, skiprows=5, names=column_names, sep=" ")
                
                if (file_num == 1) or (file_num == 2):
                    testing_dfs[key] = df
                else:
                    training_dfs[key] = df
            else: 
                df = pd.read_csv(file, skiprows=5, names=column_names)
                
                if (file_num == 1) or (file_num == 2) or (file_num == 3):
                    testing_dfs[key] = df
                else:
                    training_dfs[key] = df

In [17]:
columns_six = column_names[1:]

training_features = []

for dirfile1, its_df1 in training_dfs.items():
    a_instance = {}
    a_instance["dirfile_name"] = dirfile1 # to keep track of where the numbers are coming from

    for i, a_column in enumerate(columns_six, start=1):
        a_instance[f"min_{i}"] = its_df1[a_column].min()
        a_instance[f"max_{i}"] = its_df1[a_column].max()
        a_instance[f"mean_{i}"] = its_df1[a_column].mean()
        a_instance[f"median_{i}"] = its_df1[a_column].median()
        a_instance[f"std_dev_{i}"] = its_df1[a_column].std()
        a_instance[f"first_quart_{i}"] = its_df1[a_column].quantile(0.25)
        a_instance[f"third_quart_{i}"] = its_df1[a_column].quantile(0.75)

    training_features.append(a_instance)

In [18]:
testing_features = []

for dirfile2, its_df2 in testing_dfs.items():
    b_instance = {}
    b_instance["dirfile_name"] = dirfile2

    for j, b_column in enumerate(columns_six, start=1):
        b_instance[f"min_{j}"] = its_df2[b_column].min()
        b_instance[f"max_{j}"] = its_df2[b_column].max()
        b_instance[f"mean_{j}"] = its_df2[b_column].mean()
        b_instance[f"median_{j}"] = its_df2[b_column].median()
        b_instance[f"std_dev_{j}"] = its_df2[b_column].std()
        b_instance[f"first_quart_{j}"] = its_df2[b_column].quantile(0.25)
        b_instance[f"third_quart_{j}"] = its_df2[b_column].quantile(0.75)

    testing_features.append(b_instance)

In [19]:
all_features = training_features + testing_features
all_features_df = pd.DataFrame(all_features)
af_df = all_features_df.drop(columns=["dirfile_name"])
std_of_features = af_df.std()

In [20]:
boostrap_results = []

for a_column in af_df.columns:
    one_feat = {}
    one_feat["feat"] = a_column
    one_feat["std_dev"] = std_of_features[a_column]

    boostrap_samples = []

    for _ in range(1000):
        boostrap_sample = af_df[a_column].sample(frac=1.0, replace=True)
        boostrap_sample_std = boostrap_sample.std()
        boostrap_samples.append(boostrap_sample_std)

    bootstrap_series = pd.Series(boostrap_samples)

    left_tail = bootstrap_series.quantile(0.05)
    right_tail = bootstrap_series.quantile(0.95)

    one_feat["left_tail"] = left_tail
    one_feat["right_tail"] = right_tail

    boostrap_results.append(one_feat)


# for result in boostrap_results:
#     for key, value in result.items():
#         print(f"{key} -> {value}")
#     print("\n")

In HW3, I chose Max, Median, and Third Quartile to be the three features.

HOMEWORK 4 SECTION: